# `What are Runnables and LCEL`
---

### Intro
- A Runnable is a unit of  Work that takes Input then Process it and then process output
- Input --> Runnable --> Output
- LLMs Prompts, Chain, Retrieval --> considered as Runnables

How they can be connected together
- LLMs --> Amaming App
- invoke() --> runs a single input
- batch() --> runs multiple inputs
- stream() --> runs input in Tokens


RAG Application
- Document Loader --> Text Splitter --> Embeddings Model --> Vector Store --> Retrieval --> LLM
- All of the runnable can be connected together and we can create complex workflows using these runnables
- Complex Work


Runnable Types
1. Task Specific Runnables - Langchain core components --> LLMs, Prompt, Retrievers etc
2. Runnable Primitives -> 



RUnnable Primitives
- These are fundamental building blocks connected together for structuing exectution login in AI Workflow.
- Runnable Sequence - Run the steps in sequential order eg R1 --> R2 --> R2
- Runnable Paralle - runs the multiple steps symmetrical/ simultaneously 
- Runnable Branch - implement -> condiation exectuion
- Runnable Lamda - wraps custrom python functions into Runnables
- Runnable Passthrough - just forward input as the output


Runnable Sequence:
- it is a sequential chain of Runnables that executres one step after the other in a way such that output from one runnable will be input for the second runnable



# `Detailed Notes`

# Runnables and LCEL in LangChain

To understand modern LangChain properly, two concepts are fundamental:

* **Runnable** → the building block of a LangChain workflow.
* **LCEL (LangChain Expression Language)** → the syntax used to compose Runnables into chains.

A useful mental model is:

```text
Runnable = Building Block
LCEL     = Way to Connect Building Blocks
Chain    = Connected Workflow
```

---

# 1. What is a Runnable?

A **Runnable** is an object that represents a unit of work in LangChain.

For example:

```text
Input
  ↓
Prompt
  ↓
LLM
  ↓
Parser
  ↓
Output
```

Each of these can be represented as a Runnable.

Common Runnable types include:

```text
Runnable
├── Prompt Template
├── LLM
├── Output Parser
├── RunnableLambda
├── RunnablePassthrough
├── RunnableParallel
├── RunnableBranch
└── Retriever
```

The important idea is that these components expose a common interface.

For example:

```python
result = runnable.invoke(input)
```

So instead of learning completely different APIs for every component, LangChain gives them a common execution model.

---

# 2. Why Were Runnables Introduced?

Imagine you have:

```python
prompt
llm
parser
```

You want:

```text
prompt → llm → parser
```

Without a unified abstraction, every component could have a different API.

LangChain instead treats them as composable units:

```python
prompt | llm | parser
```

This is possible because these components implement the **Runnable interface**.

---

# 3. The Runnable Interface

A Runnable provides several important execution methods.

The most important ones are:

### `invoke()`

Run one input.

```python
result = chain.invoke({
    "topic": "Python"
})
```

Conceptually:

```text
One Input
   ↓
Runnable
   ↓
One Output
```

---

### `batch()`

Process multiple inputs.

```python
results = chain.batch([
    {"topic": "Python"},
    {"topic": "Java"},
    {"topic": "Go"}
])
```

Conceptually:

```text
Input 1 ──┐
Input 2 ──┼──→ Chain ──→ Outputs
Input 3 ──┘
```

This is useful when processing many independent requests.

---

### `stream()`

Stream output incrementally.

```python
for chunk in chain.stream({
    "topic": "Python"
}):
    print(chunk)
```

Instead of waiting for:

```text
Complete Response
```

you can receive:

```text
Python
Python is
Python is a
Python is a programming
Python is a programming language...
```

This is especially useful for chat applications.

---

### `ainvoke()`

Asynchronous execution.

```python
result = await chain.ainvoke({
    "topic": "Python"
})
```

Useful when building async applications with FastAPI or other asynchronous systems.

---

# 4. What is LCEL?

**LCEL = LangChain Expression Language.**

It provides a declarative way to compose Runnables.

The most recognizable LCEL syntax is:

```python
chain = prompt | llm | parser
```

The `|` operator means:

```text
Output of the left component
            ↓
Input of the right component
```

So:

```python
prompt | llm | parser
```

means:

```text
Prompt
  ↓
LLM
  ↓
Parser
```

---

# 5. Simple Example

Install:

```bash
pip install langchain langchain-openai
```

Then:

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple terms."
)

parser = StrOutputParser()

chain = prompt | llm | parser
```

Now:

```python
result = chain.invoke({
    "topic": "Machine Learning"
})

print(result)
```

The workflow is:

```text
{"topic": "Machine Learning"}
             ↓
       ChatPromptTemplate
             ↓
            LLM
             ↓
      StrOutputParser
             ↓
        String Output
```

This is LCEL.

---

# 6. What Exactly Happens with `|`?

When you write:

```python
chain = prompt | llm | parser
```

LangChain creates a composed Runnable.

Conceptually:

```python
chain = RunnableSequence(
    prompt,
    llm,
    parser
)
```

So this:

```python
prompt | llm | parser
```

is essentially a concise way to construct a sequential Runnable workflow.

---

# 7. RunnableSequence

A sequential workflow can also be explicitly represented using `RunnableSequence`.

```python
from langchain_core.runnables import RunnableSequence

chain = RunnableSequence(
    prompt,
    llm,
    parser
)
```

But LCEL is much cleaner:

```python
chain = prompt | llm | parser
```

Therefore, you'll frequently see the `|` syntax in modern LangChain code.

---

# 8. RunnableLambda

Sometimes your workflow needs custom Python logic.

For example:

```python
def uppercase(text):
    return text.upper()
```

You can convert that function into a Runnable:

```python
from langchain_core.runnables import RunnableLambda

uppercase_runnable = RunnableLambda(uppercase)
```

Now you can use it inside an LCEL chain:

```python
chain = prompt | llm | parser | uppercase_runnable
```

Workflow:

```text
Prompt
 ↓
LLM
 ↓
Parser
 ↓
Python Function
 ↓
Uppercase Output
```

This is powerful because you can combine:

```text
Python Logic
+
LangChain Components
+
LLMs
```

inside one workflow.

---

# 9. RunnablePassthrough

`RunnablePassthrough` means:

> Pass the input through without modifying it.

Example:

```python
from langchain_core.runnables import RunnablePassthrough
```

Suppose the input is:

```python
{
    "question": "What is RAG?"
}
```

You can preserve the original value:

```python
chain = RunnablePassthrough()
```

Then:

```python
result = chain.invoke("Hello")
```

returns:

```text
Hello
```

It becomes especially useful when constructing dictionaries.

---

# 10. RunnablePassthrough in RAG

This is where it becomes much more useful.

Suppose you have:

```text
Question
   ↓
Retriever
   ↓
Documents
```

You want to provide both:

* retrieved documents
* original question

to your prompt.

You can write:

```python
chain = {
    "context": retriever,
    "question": RunnablePassthrough()
}
```

Conceptually:

```text
                 ┌──→ Retriever ──→ context
Question ────────┤
                 └──→ Passthrough → question
```

The result might look like:

```python
{
    "context": [...retrieved documents...],
    "question": "What is RAG?"
}
```

Then:

```python
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | parser
)
```

This is one of the most important LCEL patterns for RAG.

---

# 11. RunnableParallel

You can execute multiple Runnable branches from the same input.

```python
from langchain_core.runnables import RunnableParallel

parallel = RunnableParallel(
    summary=summary_chain,
    keywords=keyword_chain
)
```

Input:

```python
{
    "topic": "Artificial Intelligence"
}
```

Output:

```python
{
    "summary": "...",
    "keywords": "..."
}
```

Architecture:

```text
                 ┌──→ Summary Chain
Input ───────────┤
                 └──→ Keyword Chain
```

This is **parallel composition**.

---

# 12. RunnableBranch

`RunnableBranch` allows conditional routing.

```python
from langchain_core.runnables import RunnableBranch
```

Example:

```python
branch = RunnableBranch(
    (
        lambda x: x["type"] == "technical",
        technical_chain
    ),
    (
        lambda x: x["type"] == "billing",
        billing_chain
    ),
    general_chain
)
```

Architecture:

```text
                   ┌── technical → Technical Chain
                   │
Input → Condition ─┼── billing ──→ Billing Chain
                   │
                   └── otherwise → General Chain
```

This is useful for:

* Query routing
* RAG routing
* Customer support
* Agent workflows
* Intent classification

---

# 13. Runnables Can Be Combined

This is the real power.

Suppose:

```python
prompt
llm
parser
```

are Runnables.

You can create:

```python
chain = prompt | llm | parser
```

Then use that chain itself as a Runnable.

For example:

```python
summary_chain = summary_prompt | llm | parser

keyword_chain = keyword_prompt | llm | parser
```

Now these complete chains can be used inside another Runnable:

```python
parallel = RunnableParallel(
    summary=summary_chain,
    keywords=keyword_chain
)
```

So:

> **A chain is itself a Runnable.**

This concept is extremely important.

---

# 14. Runnable Composition

Think of it like LEGO.

Individual pieces:

```text
Prompt
LLM
Parser
Retriever
Python Function
```

are Runnables.

You combine them:

```text
Prompt → LLM → Parser
```

Now you have a Runnable.

Then combine that Runnable with another:

```text
                    ┌──→ Chain A
Input → Parallel ────┼──→ Chain B
                    └──→ Chain C
```

That entire parallel workflow is also a Runnable.

You can keep composing.

---

# 15. LCEL Operators

The most common LCEL composition operators/patterns are:

### Sequential composition

```python
a | b
```

Meaning:

```text
a → b
```

---

### Dictionary / parallel composition

```python
{
    "a": chain_a,
    "b": chain_b
}
```

This is commonly interpreted as parallel runnable mapping.

Equivalent explicit form:

```python
RunnableParallel(
    a=chain_a,
    b=chain_b
)
```

---

### Conditional composition

```python
RunnableBranch(...)
```

Used when routing depends on conditions.

---

# 16. LCEL Example: Text Summarization

Let's build a complete small example.

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_template(
    """
    Summarize the following text in 3 bullet points:

    {text}
    """
)

parser = StrOutputParser()

chain = prompt | llm | parser
```

Run:

```python
result = chain.invoke({
    "text": """
    Artificial intelligence allows computers to perform
    tasks that traditionally require human intelligence.
    """
})

print(result)
```

Here:

```text
ChatPromptTemplate
        ↓
     ChatOpenAI
        ↓
  StrOutputParser
```

All three are Runnables.

And:

```python
prompt | llm | parser
```

is LCEL.

---

# 17. LCEL Example: RAG

Now consider a typical RAG architecture.

```text
User Question
      ↓
   Retriever
      ↓
Retrieved Documents
      ↓
     Prompt
      ↓
      LLM
      ↓
    Parser
```

Using LCEL:

```python
rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)
```

This single expression represents a fairly sophisticated workflow.

Input:

```python
"What is our refund policy?"
```

Then:

```text
                  ┌──→ Retriever ──→ context
Question ─────────┤
                  └──→ Passthrough → question
                           ↓
                         Prompt
                           ↓
                          LLM
                           ↓
                         Parser
```

This is one of the reasons understanding Runnables is essential before learning advanced LangChain RAG.

---

# 18. Runnables and RAG

A modern RAG application can be thought of as a Runnable graph:

```text
                 ┌──────── Retriever
                 │
Question ────────┼──────── Passthrough
                 │
                 ↓
              Prompt
                 ↓
                LLM
                 ↓
              Parser
```

And each component follows the Runnable abstraction.

This makes the workflow composable.

---

# 19. Runnable Methods

A practical summary:

| Method      | Purpose                 |
| ----------- | ----------------------- |
| `invoke()`  | Execute one input       |
| `batch()`   | Execute multiple inputs |
| `stream()`  | Stream output           |
| `ainvoke()` | Async single execution  |
| `abatch()`  | Async batch execution   |
| `astream()` | Async streaming         |

Example:

```python
chain.invoke(input)
```

One request.

```python
chain.batch([input1, input2, input3])
```

Multiple requests.

```python
chain.stream(input)
```

Streaming.

---

# 20. Why Runnables Matter in Production

Runnables provide a common abstraction for:

### Composition

```text
Prompt → LLM → Parser
```

### Parallelism

```text
       ┌→ Chain A
Input ─┼→ Chain B
       └→ Chain C
```

### Routing

```text
Input → Router → Appropriate Chain
```

### Streaming

```text
LLM → stream → UI
```

### Batch processing

```text
100 Inputs → Chain → 100 Outputs
```

### Async execution

```text
Application
     ↓
async Runnable
```

This makes LangChain workflows easier to compose and integrate into applications.

---

# 21. LCEL vs Traditional Chains

You may encounter older LangChain tutorials using things such as:

```python
LLMChain
SimpleSequentialChain
SequentialChain
```

Modern LangChain generally favors **Runnable-based composition and LCEL** for constructing these workflows.

For example, older style:

```python
LLMChain(
    llm=llm,
    prompt=prompt
)
```

Modern style:

```python
chain = prompt | llm
```

The modern approach is more composable because the resulting object follows the Runnable interface.

---

# 22. Runnable vs LCEL

This distinction is important for interviews.

### Runnable

A **Runnable is the abstraction/interface for an executable LangChain component or workflow.**

Examples:

```text
LLM
Prompt
Retriever
Parser
RunnableLambda
RunnableParallel
RunnableBranch
```

### LCEL

**LCEL is the expression syntax used to compose Runnables.**

Example:

```python
chain = prompt | llm | parser
```

So:

```text
Runnable
   ↓
Building Block

LCEL
   ↓
Composition Language

Chain
   ↓
Composed Runnable Workflow
```

---

# 23. The Complete Picture

You can now connect the concepts you've learned:

```text
                         LangChain
                             │
                             ▼
                         Runnables
                             │
              ┌──────────────┼──────────────┐
              │              │              │
           Prompt           LLM          Retriever
              │              │              │
              └──────────────┼──────────────┘
                             │
                             ▼
                           LCEL
                             │
                  ┌──────────┼──────────┐
                  │          │          │
               Sequential  Parallel  Conditional
                  │          │          │
                  ▼          ▼          ▼
               A → B → C   A + B + C   A/B/C
                  │          │          │
                  └──────────┼──────────┘
                             ▼
                      Application Workflow
```

## The most important mental model

If you're learning LangChain for **RAG and GenAI engineering**, remember this progression:

```text
Runnable
   ↓
LCEL
   ↓
Sequential Composition
   ↓
Parallel Composition
   ↓
Conditional Routing
   ↓
RAG Chains
   ↓
Agents / Complex Workflows
```

Once you understand **Runnables + LCEL**, many LangChain concepts that initially look unrelated become variations of the same composition model.
